# مولد فيديوهات بسيط للكورسات
هذا Notebook بسيط لإنشاء فيديوهات من نصوص الكورسات الموجودة

In [ ]:
# تثبيت الحزم الأساسية
!pip install gTTS moviepy pillow
!pip install google-cloud-texttospeech

In [ ]:
import os
from pathlib import Path
from gtts import gTTS
from moviepy.editor import *
from PIL import Image, ImageDraw, ImageFont

# إعداد المجلدات
SCRIPTS_DIR = Path('video-scripts')
OUTPUT_DIR = Path('videos')
OUTPUT_DIR.mkdir(exist_ok=True)

print("تم إعداد المجلدات بنجاح")

In [ ]:
def read_script(file_path):
    """قراءة ملف النص"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

def text_to_speech(text, output_path):
    """تحويل النص إلى صوت"""
    tts = gTTS(text=text, lang='ar')
    tts.save(output_path)
    return output_path

def create_simple_slide(text, output_path):
    """إنشاء شريحة بسيطة"""
    img = Image.new('RGB', (1920, 1080), color='#0A1628')
    draw = ImageDraw.Draw(img)
    
    # إضافة نص بسيط
    draw.text((100, 100), text[:100], fill='#00D4AA')
    
    img.save(output_path)
    return output_path

In [ ]:
def generate_video(script_file):
    """إنشاء فيديو من ملف نص"""
    script_path = Path(script_file)
    
    # قراءة النص
    text = read_script(script_path)
    
    # إنشاء الصوت
    audio_path = OUTPUT_DIR / f"{script_path.stem}.mp3"
    text_to_speech(text, audio_path)
    
    # إنشاء شريحة
    slide_path = OUTPUT_DIR / f"{script_path.stem}.png"
    create_simple_slide(text, slide_path)
    
    # إنشاء الفيديو
    audio = AudioFileClip(str(audio_path))
    slide = ImageClip(str(slide_path)).set_duration(audio.duration)
    video = slide.set_audio(audio)
    
    # حفظ الفيديو
    output_path = OUTPUT_DIR / f"{script_path.stem}.mp4"
    video.write_videofile(str(output_path), fps=24)
    
    return output_path

In [ ]:
# إنشاء فيديو لملف واحد
script_file = 'video-scripts/capa-1-1.md'
video_path = generate_video(script_file)
print(f"تم إنشاء الفيديو: {video_path}")

In [ ]:
# إنشاء فيديوهات لجميع الملفات
script_files = list(Path('video-scripts').glob('*.md'))

for script_file in script_files[:5]:  # أول 5 ملفات للتجربة
    try:
        video_path = generate_video(script_file)
        print(f"✅ {script_file.name} -> {video_path}")
    except Exception as e:
        print(f"❌ {script_file.name}: {e}")